<a href="https://colab.research.google.com/github/KalaiselvamK23/MS-Elevate-Microsoft-Azure/blob/main/Sleep_efficiency%2Cloaded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter


file_path = "Sleep_Efficiency.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "equilibriumm/sleep-efficiency",
  file_path,

  # documenation for more information:
  # https://github.com/Kaggle/kaggle/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/tmp/ipython-input-1201118534.py:11: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'sleep-efficiency' dataset.
First 5 records:    ID  Age  Gender              Bedtime          Wakeup time  Sleep duration  \
0   1   65  Female  2021-03-06 01:00:00  2021-03-06 07:00:00             6.0   
1   2   69    Male  2021-12-05 02:00:00  2021-12-05 09:00:00             7.0   
2   3   40  Female  2021-05-25 21:30:00  2021-05-25 05:30:00             8.0   
3   4   40  Female  2021-11-03 02:30:00  2021-11-03 08:30:00             6.0   
4   5   57    Male  2021-03-13 01:00:00  2021-03-13 09:00:00             8.0   

   Sleep efficiency  REM sleep percentage  Deep sleep percentage  \
0              0.88                    18                     70   
1              0.66                    19                     28   
2              0.89                    20                     70   
3              0.51                    23                     25   
4              0.76                    27                     55   

   Light sleep percent

In [ ]:

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


data = df.copy()

# Encode Categorical
le = LabelEncoder()
data['Smoking status_encoded'] = le.fit_transform(data['Smoking status'])
data['Gender_encoded'] = le.fit_transform(data['Gender'])


data.drop(['Smoking status', 'Gender', 'Bedtime', 'Wakeup time'], axis=1, inplace=True)

# 2. Split Features and Target
X = data[['Age','Gender_encoded','REM sleep percentage','Deep sleep percentage','Light sleep percentage','Awakenings','Smoking status_encoded','Caffeine consumption','Alcohol consumption','Exercise frequency']]
y = data['Sleep efficiency'] # 1D Series is better for Sklearn

# 3. Impute & Scale
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

train_x, test_x, train_y, test_y = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
train_x_scaled = scaler.fit_transform(train_x)
test_x_scaled = scaler.transform(test_x)

# 4. Train Best Model (Random Forest usually performs best here)
final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(train_x_scaled, train_y)

print("Model Training Complete.")

Model Training Complete.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create input fields
age_in = widgets.IntSlider(value=30, min=10, max=80, description='Age:')
gender_in = widgets.Dropdown(options=[('Male', 1), ('Female', 0)], description='Gender:')
rem_in = widgets.FloatSlider(value=20, min=0, max=100, description='REM %:')
deep_in = widgets.FloatSlider(value=50, min=0, max=100, description='Deep %:')
light_in = widgets.FloatSlider(value=30, min=0, max=100, description='Light %:')
awake_in = widgets.IntSlider(value=1, min=0, max=10, description='Awakenings:')
smoke_in = widgets.Dropdown(options=[('Yes', 1), ('No', 0)], description='Smoker:')
caff_in = widgets.FloatSlider(value=0, min=0, max=500, description='Caffeine (mg):')
alc_in = widgets.FloatSlider(value=0, min=0, max=10, description='Alcohol (oz):')
exe_in = widgets.IntSlider(value=3, min=0, max=7, description='Exercise/wk:')

button = widgets.Button(description="Predict Sleep Efficiency", button_style='success')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        # Arrange inputs for the model
        user_data = np.array([[age_in.value, gender_in.value, rem_in.value, deep_in.value,
                               light_in.value, awake_in.value, smoke_in.value,
                               caff_in.value, alc_in.value, exe_in.value]])


        user_scaled = scaler.transform(user_data)

        # Predict
        prediction = final_model.predict(user_scaled)
        print(f"--- PREDICTION ---")
        print(f"Estimated Sleep Efficiency: {prediction[0]:.2f}")


        if prediction[0] > 0.85:
            print("Status: Excellent Sleep Quality! 🌙")
        elif prediction[0] > 0.70:
            print("Status: Good Sleep Quality. 👍")
        else:
            print("Status: Poor Sleep Quality. Consider adjusting habits. ⚠️")

button.on_click(on_button_clicked)

# Layout
ui = widgets.VBox([
    widgets.HBox([age_in, gender_in]),
    widgets.HBox([rem_in, deep_in, light_in]),
    widgets.HBox([awake_in, smoke_in]),
    widgets.HBox([caff_in, alc_in, exe_in]),
    button, output
])

display(ui)